In [1]:
import os
import xarray as xr
import numpy as np

In [2]:
# === Filename builders ===
def build_arise_path(ens_num, date_range):
    base = (
        "/glade/campaign/cesm/collections/ARISE-SAI-1.5/"
        f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{ens_num}.cam.h4.O3_SRF.{date_range}.nc"
    )
    return base


def build_ssp245_path(ens_num, date_range):
    base = (
        "/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/"
        f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{ens_num}.cam.h4.O3_SRF.{date_range}.nc"
    )
    return base


def build_hist_path(ens_num, date_range):
    base = (
        "/glade/campaign/cesm/development/wawg/WACCM6-TSMLT-HIST/"
        f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{ens_num}.cam.h4.O3_SRF.{date_range}.nc"
    )
    return base

In [3]:
# === File list generator ===
def get_file_list(scenario, ens_num):
    files = []
    num = f"{ens_num:02d}"

    if scenario == "ARISE":
        if ens_num in [5, 8, 9]:  # These files are saved yearly
            for year in range(2035, 2070 if ens_num in [8, 9] else 2069):
                start = f"{year}010100"
                end = f"{year+1}010100"
                date_range = f"{start}-{end}"
                files.append(build_arise_path(num, date_range))
            if ens_num in [5]:
                files.append(build_arise_path(num, "2069010100-2069123100"))
        else:  # These files are saved in decades
            for start, end in [(2035, 2045), (2045, 2055), (2055, 2065), (2065, 2069)]:
                date_range = (
                    f"{start}010100-{end}123100"
                    if start == 2065
                    else f"{start}010100-{end}010100"
                )
                files.append(build_arise_path(num, date_range))

    elif scenario == "SSP245":
        if ens_num <= 5:  # saved through 2100
            for start, end in [(2015, 2025), (2025, 2035), (2035, 2045), (2045, 2055), (2055, 2065), (2065, 2075)]:
                date_range = f"{start}010100-{end}010100"
                files.append(build_ssp245_path(num, date_range))
        else:  # ends in 20691231
            for start, end in [(2015, 2025), (2025, 2035), (2035, 2045), (2045, 2055), (2055, 2065), (2065, 2069)]:
                date_range = (
                    f"{start}010100-{end}123100"
                    if start == 2065
                    else f"{start}010100-{end}010100"
                )
                files.append(build_ssp245_path(num, date_range))

    elif scenario == "hist":
        files.append(build_hist_path(num, "1988010100-1998010100"))
        files.append(build_hist_path(num, "1998010100-1999123100"))
        files.append(build_hist_path(num, "2000010100-2010010100"))

    return files

In [4]:
# === Processing function ===
def calculate_monthly_mean_8hrdailymax(start_date, end_date, o3_surf):
    daterange = xr.date_range(start_date, end_date, calendar="noleap", use_cftime=True)

    MDA8 = xr.DataArray(  # Maximum Daily 8hr Average O3
        np.nan,
        dims=["time", "lat", "lon"],
        coords={"time": daterange, "lat": o3_surf.lat, "lon": o3_surf.lon},
    )

    for i in range(len(daterange)):
        date = daterange[i].strftime("%Y-%m-%d")
        o3_day = o3_surf.sel(time=slice(date + " 00:00:00", date + " 23:00:00"))
        o3_rolling = o3_day.rolling(time=8).mean()
        MDA8[i, :, :] = o3_rolling.max("time")

    monthly_mean = MDA8.resample(time="ME").mean()
    return monthly_mean

## Historical

In [16]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/ozone/MDA8/"
SCENARIOS = ["hist"]
ens_num = 1

# === Main loop ===
for scenario in SCENARIOS:
    print(f"Processing {scenario}")
    file_list = get_file_list(scenario, ens_num)
    monthly_means = []

    for f in file_list:
        if not os.path.exists(f):
            raise ValueError(f"Missing: {f}")

        print(f"Reading {os.path.basename(f)}")
        ds = xr.open_dataset(f)["O3_SRF"]

        start_date = str(ds.time[0].values)[:10]
        end_date = str(ds.time[-1].values)[:10]
        end_time = str(ds.time[-1].values)[11:16]
        midnight = "00:00"

        if end_time == midnight:
            print("changing final time step")
            end_date = str(ds.time[-2].values)[:10]

        mm = calculate_monthly_mean_8hrdailymax(start_date, end_date, ds)

        if scenario == "hist":
            # Trim to 1990-2009
            mm = mm.sel(time=slice("1990-01-01", "2009-12-31"))

        monthly_means.append(mm)

    if monthly_means:
        combined = xr.concat(monthly_means, dim="time")

        if scenario == "hist":
            dates = "19900101-20091231"

        out_file = f"MDA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        combined.to_netcdf(out_path)

print("All processing complete.")


Processing hist
Reading b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h4.O3_SRF.1988010100-1998010100.nc
Reading b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h4.O3_SRF.1998010100-1999123100.nc
changing final time step
Reading b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h4.O3_SRF.2000010100-2010010100.nc
Saving to /glade/work/awells/air_quality/CESM/MDA8/MDA8_CESM2_hist_01_1990010-20091231.nc
All processing complete.


## Future Scenarios

In [8]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/ozone/MDA8/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        file_list = get_file_list(scenario, ens_num)
        monthly_means = []

        for f in file_list:
            if not os.path.exists(f):
                raise ValueError(f"Missing: {f}")

            print(f"Reading {os.path.basename(f)}")
            ds = xr.open_dataset(f)["O3_SRF"]

            start_date = str(ds.time[0].values)[:10]
            end_date = str(ds.time[-1].values)[:10]
            end_time = str(ds.time[-1].values)[11:16]
            midnight = "00:00"

            if end_time == midnight:
                print("changing final time step")
                end_date = str(ds.time[-2].values)[:10]

            mm = calculate_monthly_mean_8hrdailymax(start_date, end_date, ds)

            if scenario == "SSP245":
                # Trim to 2069-12-31
                mm = mm.sel(time=slice("2020-01-01", "2069-12-31"))

            if scenario == "ARISE" and ens_num in [8, 9]:
                # Trim to 2069-12-31
                mm = mm.sel(time=slice("2035-01-01", "2069-12-31"))

            monthly_means.append(mm)

        if monthly_means:
            combined = xr.concat(monthly_means, dim="time")

            if scenario == "ARISE":
                dates = "20350101-20691231"
            elif scenario == "SSP245":
                dates = "20200101-20691231"

            out_file = f"MDA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving to {out_path}")
            combined.to_netcdf(out_path)

print("All processing complete.")


Processing SSP245, Ensemble 07
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.007.cam.h4.O3_SRF.2015010100-2025010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.007.cam.h4.O3_SRF.2025010100-2035010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.007.cam.h4.O3_SRF.2035010100-2045010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.007.cam.h4.O3_SRF.2045010100-2055010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.007.cam.h4.O3_SRF.2055010100-2065010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.007.cam.h4.O3_SRF.2065010100-2069123100.nc
changing final time step
Saving to /glade/work/awells/air_quality/CESM/ozone/MDA8/MDA8_CESM2_SSP245_07_20200101-20691231.nc
Processing SSP245, Ensemble 08
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.008.cam.h4.O3_SRF.2015010100-2025010100.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.008.cam.h4.O3_SRF.2025010100-2035010100.nc
Reading b.e21.